## ------------------------------------------------------------ ###
## Set up environment
## ------------------------------------------------------------ ###

In [1]:
import os
import sys
import glob
import rasterio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

sys.path.append('../ProcessEvents/')
from config import CATCHMENT_LOOKUP_DICT, OUT_DIR 

event_details_fp = 'EventDetails_v5'
flood_5km_results_fp = "5km_total_v5"

## ------------------------------------------------------------ ###
## Get list of catchments to process
## ------------------------------------------------------------ ###

In [2]:
all_catchments = set(CATCHMENT_LOOKUP_DICT.keys())

catchments_with_flood_output = []
for catchment_num in all_catchments:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}.pkl"
    fp2 = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}_new.pkl"
    flood_fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/PluvialResults/{flood_5km_results_fp}/Catchment_{catchment_num}/Ens01_{catchment_num}/10cm/flooded_area_5km_total_Ens01_{catchment_num}_10cm.nc"
        
    if os.path.isfile(flood_fp) and (os.path.isfile(fp) or os.path.isfile(fp2)):
        catchments_with_flood_output.append(catchment_num)
    else:
        pass
        #print(catchment_num)
        #print(os.path.isfile(fp))
        #print(os.path.isfile(flood_fp))

## ------------------------------------------------------------ ###
## Join data for all catchments
## ------------------------------------------------------------ ###

Account for fact some of the catchments I have run the new processing events which adds antecedent conditions and changes the way dates are stored

In [3]:
rainfall_events_all = []
for catchment_num in catchments_with_flood_output:
    catchment_name = CATCHMENT_LOOKUP_DICT[catchment_num]
    if catchment_num in ['33_b', '26', '89', '46']:
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}_new.pkl"
    else:
        fp = f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/Catchment_{catchment_num}/{catchment_name}.pkl"
    rainfall_events = pd.read_pickle(fp)
    if 'threshold_ante_rain_1d_mean_t80' in rainfall_events.columns:
        print("Contains antecedent rainfall variables", catchment_num)
        rainfall_events.drop(['threshold_ante_rain_1d_mean_t80', 'cluster_ante_rain_1d_mean_t80',
       'threshold_ante_rain_2d_mean_t80', 'cluster_ante_rain_2d_mean_t80',
       'threshold_ante_rain_5d_mean_t80', 'cluster_ante_rain_5d_mean_t80',
       'threshold_ante_rain_10d_mean_t80', 'cluster_ante_rain_10d_mean_t80',
       'threshold_ante_sm_mean_t80', 'cluster_ante_sm_mean_t80',
                                   'threshold_ante_rain_1d_mean_t60', 'cluster_ante_rain_1d_mean_t60',
       'threshold_ante_rain_2d_mean_t60', 'cluster_ante_rain_2d_mean_t60',
       'threshold_ante_rain_5d_mean_t60', 'cluster_ante_rain_5d_mean_t60',
       'threshold_ante_rain_10d_mean_t60', 'cluster_ante_rain_10d_mean_t60',
                                   'threshold_ante_rain_2d_mean_t50', 'cluster_ante_rain_2d_mean_t50',
       'threshold_ante_rain_5d_mean_t50', 'cluster_ante_rain_5d_mean_t50',
       'threshold_ante_rain_10d_mean_t50', 'cluster_ante_rain_10d_mean_t50',
                            'neighbourhood_ante_rain_2d_mean', 'neighbourhood_ante_rain_2d_point',
       'neighbourhood_ante_rain_5d_mean', 'neighbourhood_ante_rain_5d_point',
       'neighbourhood_ante_rain_10d_mean', 'neighbourhood_ante_rain_10d_point',
                     'threshold_ante_sm_mean_t60', 'cluster_ante_sm_mean_t60',
                            'threshold_ante_rain_1d_mean_t50', 'cluster_ante_rain_1d_mean_t50',
       'threshold_ante_sm_mean_t50', 'cluster_ante_sm_mean_t50',      'neighbourhood_ante_rain_1d_mean', 'neighbourhood_ante_rain_1d_point',
       'neighbourhood_ante_sm_mean', 'neighbourhood_ante_sm_point',], axis=1, inplace=True)
    if 'start_month' in rainfall_events.columns:
        print("contains start_month", catchment_num)
        del rainfall_events['start_day']
        del rainfall_events['start_hour']
        rainfall_events.rename(columns={'start_month': 'month'}, inplace=True)
        rainfall_events.rename(columns={'start_year':'start_year'}, inplace=True)
    rainfall_events['catchment_num'] = catchment_num
    # print(f"Catchment {catchment_num} has {len(rainfall_events)}, of which {len(rainfall_events) - len(rainfall_events_complete)} are mismatched")
    rainfall_events_all.append(rainfall_events) 
rainfall_events_all_df = pd.concat(rainfall_events_all, ignore_index=True)   
# rainfall_events_all_df = rainfall_events_all_df[rainfall_events_all_df['max_precip']<130].copy()

Contains antecedent rainfall variables 40
contains start_month 40
Contains antecedent rainfall variables 105
contains start_month 105
Contains antecedent rainfall variables 23
contains start_month 23


In [4]:
len(rainfall_events_all_df.loc[~rainfall_events_all_df["catchment_num"].map(lambda x: isinstance(x, (int, float))), "catchment_num"].unique())

3

## ------------------------------------------------------------ ###
## Save to pickle (can't save to csv as this cuts off some of the data)
## ------------------------------------------------------------ ###

In [5]:
rainfall_events_all_df.to_pickle(f"/scratch/hydro4/users/kv25483/FutureFlood/Data/{event_details_fp}/all_catchments.pkl")